# **Model Evaluation**

# 1. Create the evaluation notebook and reload our saved models

In [1]:
import pandas as pd
import joblib

# Reload everything we saved across Phases 5-7
regression_model = joblib.load("../src/regression_model.pkl")
classification_model = joblib.load("../src/classification_model.pkl")
label_encoder = joblib.load("../src/label_encoder.pkl")
item_similarity = joblib.load("../src/item_similarity.pkl")
content_similarity = joblib.load("../src/content_similarity.pkl")

print("Regression model loaded:", type(regression_model).__name__)
print("Classification model loaded:", type(classification_model).__name__)
print("Item similarity matrix shape:", item_similarity.shape)
print("Content similarity matrix shape:", content_similarity.shape)

Regression model loaded: RandomForestRegressor
Classification model loaded: RandomForestClassifier
Item similarity matrix shape: (30, 30)
Content similarity matrix shape: (1698, 1698)


# 2. Build one consolidated comparison table

In [2]:
# Consolidated results from everything we tested in Phases 5-7
results_summary = pd.DataFrame([
    # Regression models
    {"Task": "Regression", "Model": "Linear Regression", "Metric": "R²", "Score": 0.0767, "Notes": "Baseline"},
    {"Task": "Regression", "Model": "Random Forest (unconstrained)", "Metric": "R²", "Score": -0.0268, "Notes": "Overfit - discarded"},
    {"Task": "Regression", "Model": "Random Forest (constrained)", "Metric": "R²", "Score": 0.0955, "Notes": "Final choice"},
    
    # Classification models
    {"Task": "Classification", "Model": "Random Forest (balanced)", "Metric": "Accuracy", "Score": 0.31, "Notes": "Final choice - fair to rare classes"},
    {"Task": "Classification", "Model": "Random Forest (unbalanced)", "Metric": "Accuracy", "Score": 0.47, "Notes": "Higher accuracy but ignores rare classes"},
    
    # Recommendation systems
    {"Task": "Recommendation", "Model": "Collaborative Filtering (item-based)", "Metric": "Coverage", "Score": 30, "Notes": "Only 30/1698 attractions have rating history"},
    {"Task": "Recommendation", "Model": "Content-Based Filtering", "Metric": "Coverage", "Score": 1698, "Notes": "Covers full catalog"},
])

print(results_summary.to_string(index=False))

          Task                                Model   Metric     Score                                        Notes
    Regression                    Linear Regression       R²    0.0767                                     Baseline
    Regression        Random Forest (unconstrained)       R²   -0.0268                          Overfit - discarded
    Regression          Random Forest (constrained)       R²    0.0955                                 Final choice
Classification             Random Forest (balanced) Accuracy    0.3100          Final choice - fair to rare classes
Classification           Random Forest (unbalanced) Accuracy    0.4700     Higher accuracy but ignores rare classes
Recommendation Collaborative Filtering (item-based) Coverage   30.0000 Only 30/1698 attractions have rating history
Recommendation              Content-Based Filtering Coverage 1698.0000                          Covers full catalog


# 3. Save the comparison table as a file

In [3]:
# Save as Excel (easy to open, copy into a report, or attach as a deliverable)
results_summary.to_excel("../notebooks/model_comparison_summary.xlsx", index=False)

print("Saved model_comparison_summary.xlsx")

Saved model_comparison_summary.xlsx


# 4. final interpretation

In [4]:
final_evaluation = """
# Final Model Evaluation & Comparison

## Regression (Predicting Rating)
Best model: Random Forest, constrained (max_depth=8, min_samples_leaf=20) - R² = 0.0955
- Outperformed both Linear Regression and an unconstrained Random Forest (which overfit
  badly and produced negative R² on test data).
- Overall predictive power is modest across all approaches tried, indicating that
  demographic/attraction-type features alone explain only a small portion of what
  drives individual satisfaction ratings.

## Classification (Predicting Visit Mode)
Best model: Random Forest with class_weight='balanced' - 31% accuracy
- A second, unbalanced version reached 47% accuracy but completely failed to identify
  rare classes (Business, Solo) - 0% recall on both.
- Chose the balanced model as final, prioritizing the project's core goal (segmenting
  ALL traveler types for targeted marketing) over raw accuracy.
- This reflects a deliberate accuracy-vs-fairness trade-off, not a modeling failure.

## Recommendation System
Two complementary systems were built:
- Collaborative Filtering (item-based): works only for the 30 attractions with rating
  history, but leverages real user behavior patterns.
- Content-Based Filtering: covers the full 1,698-attraction catalog using attraction
  type, filling the gap collaborative filtering can't reach.
- Together, they provide full-catalog coverage while still using behavioral data
  where it's available - a practical hybrid approach for a growing platform.

## Overall Conclusion
All three tasks (regression, classification, recommendation) are inherently limited by
the available features - individual taste, group composition, and detailed booking
context aren't captured in this dataset. Given these constraints, the final models
represent well-reasoned, defensible choices with explicitly documented trade-offs
rather than "best possible" performance in absolute terms.
"""

with open("../notebooks/final_evaluation.md", "w") as f:
    f.write(final_evaluation)

print("Saved final_evaluation.md")

Saved final_evaluation.md


In [5]:
master = pd.read_excel("../data/cleaned/master_dataset.xlsx")
print(master['Continent'].unique())
print(master['AttractionType'].unique())
print(master['Region'].unique())
print(master['Country'].unique())

<ArrowStringArray>
['Europe', 'America', 'Africa', 'Australia & Oceania', 'Asia']
Length: 5, dtype: str
<ArrowStringArray>
[       'Nature & Wildlife Areas',                    'Water Parks',
                        'Beaches',                'Religious Sites',
 'Points of Interest & Landmarks',                     'Waterfalls',
                 'National Parks',                       'Volcanos',
                  'Neighborhoods',             'Speciality Museums',
                           'Spas',                'Caverns & Caves',
          'Flea & Street Markets',                        'Ballets',
                  'Ancient Ruins',                'History Museums',
                 'Historic Sites']
Length: 17, dtype: str
<ArrowStringArray>
[  'Western Europe', 'Northern America',    'South America',
   'Central Europe',      'East Africa',  'Southern Africa',
        'Australia',          'Oceania',      'Middle East',
   'Eastern Europe',  'South East Asia',       'South Asia',
    